# AI-Assisted Momentum Reversal Risk Monitor

## 1. The PM problem

Momentum weakness is ambiguous. The same drawdown can be ordinary noise, a **recovery-driven reversal**, or a **crowded-position unwind**.

> **Central question:** Is current momentum weakness ordinary noise, a recovery-driven crash setup, or a crowded unwind — and what should the PM monitor next?

This notebook is a product walkthrough. It does **not** predict crash timing or issue a trade instruction.

**Default monitored book:** S&P 500 12-1 long-10 / short-10 (research stand-in).  
**Comparison context only:** Ken French UMD / Daniel–Moskowitz market state.


## 2. Two momentum-crash mechanisms

### Mechanism 1 — Recovery-driven momentum crash (Daniel–Moskowitz)

**PM question:** Is the market recovering from a severe drawdown in a way that could produce a sharp loser-stock rebound and damage momentum?

Watch for prior market drawdown, recovery state, loser- or short-leg rebound, beta asymmetry, short-leg losses, and momentum-portfolio drawdown.

### Mechanism 2 — Crowded-position unwind (Khandani–Lo)

**PM question:** Is a crowded momentum trade being reduced or unwound in a way that could amplify losses across similar portfolios?

Watch for crowded or concentrated exposure, unusual reductions in technology or momentum exposure, correlated selling, and possible deleveraging or liquidity pressure.

Do **not** claim forced deleveraging unless the evidence supports it.

| Mechanism | Support | Weaken |
| --- | --- | --- |
| Recovery-driven crash | Panic / severe drawdown → rapid recovery → loser rebound / short-leg pain | Soft recovery without panic; no short-basket stress |
| Crowded unwind | Crowding / concentration + synchronized selling + weak absorption | Selling without crowding; healthy absorption |


## 3. How the decision workflow works

```text
Deterministic monitors → mechanism read (DM vs KL) → evidence challenge → PM next checks
```

The PM uses the combined read to choose among:

1. maintain monitoring;
2. inspect the short leg or concentrated exposures;
3. challenge the signal with additional evidence;
4. discuss whether risk escalation is warranted.

**AI evidence layer:** organizes supporting, contradicting, and missing evidence. It does **not** generate the deterministic risk signal and cannot rewrite metrics, thresholds, triggers, or risk state.


### Setup

Change `CONFIG` in the next parameter cell, then run all cells. It controls the live `run_mvp` assessment only. The semiconductor, March 2020, January 2024, and cross-case sections are frozen product packs under `outputs/`; they are reference cases and do not change with `CONFIG`.

Supported live dates must be trading dates covered by the bundled processed data (currently through 2026-05-29).


In [ ]:
# Bootstrap imports whether the kernel starts in the repository root or notebooks/.
from pathlib import Path
import sys

from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "mvp" / "pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from " + str(start))

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.mvp.config import MVPConfig
from src.mvp.pipeline import run_mvp

print("Repository root:", ROOT)


In [ ]:
# PM sandbox parameters: edit this block, then Run All.
CONFIG = MVPConfig(
    as_of_date="2024-01-05",
    compare_to_date="2023-12-01",
    threshold_profile="default",
    horizon_days=20,
    use_llm=False,  # reliable offline path; optional LLM requires an injected provider
)

CASE_PACKS = {
    "current_semi": ROOT / "outputs" / "current_semi_unwind",
    "march_2020": ROOT / "outputs" / "march_2020_reference",
    "quiet_2024": ROOT / "outputs" / "quiet_control_2024",
    "cross_case": ROOT / "outputs" / "cross_case_comparison.md",
}

missing = [
    label
    for label, path in [
        ("current_semi/pm_case_read.md", CASE_PACKS["current_semi"] / "pm_case_read.md"),
        ("current_semi/mechanism_comparison.md", CASE_PACKS["current_semi"] / "mechanism_comparison.md"),
        ("march_2020/pm_case_read.md", CASE_PACKS["march_2020"] / "pm_case_read.md"),
        ("quiet_2024/pm_case_read.md", CASE_PACKS["quiet_2024"] / "pm_case_read.md"),
        ("cross_case_comparison.md", CASE_PACKS["cross_case"]),
    ]
    if not path.exists()
]
if missing:
    raise FileNotFoundError("Missing product packs: " + ", ".join(missing))
print("Live CONFIG:", CONFIG)
print("Frozen product packs ready.")


## 4. Current semiconductor case

**Primary demo.** Frozen assessment date **2026-05-29** (historical partial read — not a live August assessment).

Use the same reading structure for every case:

1. **Current read** — what is happening?
2. **Mechanism assessment** — which momentum-crash mechanism is supported?
3. **Portfolio implication** — where is the risk located?
4. **Evidence view** — what supports or contradicts the interpretation?
5. **PM interpretation** — what should the PM monitor next?

Expected credible conclusion:

> Localized crowding and meaningful structural pressure are supported; a broad recovery-driven crash or forced unwind is **not** yet confirmed.


In [ ]:
display(Markdown((CASE_PACKS["current_semi"] / "pm_case_read.md").read_text()))


### Mechanism lenses (frozen May 29)

Keep Daniel–Moskowitz, Khandani–Lo, and fundamental / sector repricing separate. Do not force a single winner.


In [ ]:
display(Markdown((CASE_PACKS["current_semi"] / "mechanism_comparison.md").read_text()))


## 5. AI evidence view

The AI / evidence layer answers:

- Why might this signal be occurring?
- Which mechanism does the evidence support?
- What evidence argues against the risk interpretation?
- What information should the PM check next?
- How confident should the PM be in the narrative?

Below: supporting / contradicting / not-yet-confirmed items from the frozen semi pack, plus the live quiet-control interpreter path for transparency.


In [ ]:
import re

case_read = (CASE_PACKS["current_semi"] / "pm_case_read.md").read_text()

def section(title: str, text: str) -> str:
    pattern = rf"## {re.escape(title)}\n(.*?)(?=\n## |\Z)"
    match = re.search(pattern, text, flags=re.S)
    return match.group(0).strip() if match else f"_Section not found: {title}_"

for title in [
    "What is supported",
    "What remains unconfirmed",
    "What would confirm propagation",
    "What would invalidate the current interpretation",
    "Why broad action may still be premature",
]:
    display(Markdown(section(title, case_read)))


### Parameterized live assessment — `CONFIG` through `run_mvp`

This section is recomputed from the parameter cell near the top. With the default `CONFIG` it reproduces the 2024 quiet-control date; after a parameter change it reports the selected date. It does not rewrite the frozen case packs below.


In [ ]:
result = run_mvp(CONFIG)
card = result.deterministic_input
unwind = result.unwind.to_dict()
mech = result.mechanical_unwind
interp = result.interpretation
pm = result.pm_response

scenarios = {row["scenario"]: row["status"] for row in unwind["mechanism_scenarios"]}
summary = f"""
**Live parameterized assessment ({card.as_of_date})**

| Layer | Read |
| --- | --- |
| UMD / market context | `{card.overall_risk_state}` (comparison only) |
| Scorecard triggers | {len(card.triggered_quant_signals)} |
| Recovery crash | `{scenarios.get("bear_market_recovery_crash")}` |
| Short-book reversal | `{scenarios.get("short_book_reversal_crash")}` |
| Crowded theme unwind | `{scenarios.get("crowded_theme_unwind")}` |
| Mechanical state | `{mech.unwind_state}` |
| Classification | `{unwind.get("scenario_classification")}` |

**Interpreter (constrained):** {interp.pm_interpretation if interp else "unavailable"}

**PM posture:** {pm.current_posture if pm else "unavailable"}

*{pm.why_not_act_yet if pm else ""}*
"""
display(Markdown(summary))


## 6. 2020 historical validation

Primary historical validation case (**2020-03-24**). Purpose: show that when a known momentum-reversal episode occurred, recovery-crash indicators behaved coherently — not to claim full predictive performance.

Look for the sequence: severe drawdown → rapid recovery → short-leg pain / beta asymmetry → momentum losses. Crowded unwind remains secondary / unconfirmed in this pack.


In [ ]:
display(Markdown((CASE_PACKS["march_2020"] / "pm_case_read.md").read_text()))


### Mechanism lenses (March 2020)


In [ ]:
display(Markdown((CASE_PACKS["march_2020"] / "mechanism_comparison.md").read_text()))


## 7. 2024 quiet control

Control case (**2024-01-05**). Same framework, different conclusion: incomplete recovery preconditions, no confirmed crowded unwind, contained short-leg pressure. Escalation is not justified.

This is how the product shows it is **selective rather than permanently alarmist**.


In [ ]:
display(Markdown((CASE_PACKS["quiet_2024"] / "pm_case_read.md").read_text()))


### Mechanism lenses (January 2024)


In [ ]:
display(Markdown((CASE_PACKS["quiet_2024"] / "mechanism_comparison.md").read_text()))


## 8. Cross-case comparison

One compact table across the three product cases. Values come from repository outputs, not hard-coded demo claims.


In [ ]:
display(Markdown(CASE_PACKS["cross_case"].read_text()))


## 9. Limitations and next steps

- Default PM book uses survivorship-biased current SPY membership; not a live holdings plug-in yet.
- UMD / DM state is comparison context only — never a PM-book crash probability.
- Crowding and mechanical layers are public-data proxies, not observed ownership, leverage, financing, or dealer inventory.
- Evidence is exact-date cached replay, not institutional live retrieval.
- Mechanism scenarios are descriptive rules without out-of-sample predictive validation.
- The AI layer organizes and challenges evidence; it cannot change deterministic values or triggers.
- Forced deleveraging is never inferred from correlation or turnover alone.

**Production imagination** (holdings plug-in, observed crowding, financing / flow overlays): see `Future_To_DO.md`.

**Reviewer docs:** `docs/methodology.md` · `docs/limitations.md` · `docs/demo_walkthrough.md`.
